In [ ]:
import os
import glob
import pandas as pd

# Relative path from 'master_datasets/kevin' to 'data_log/data_kev'
data_dir = os.path.join("..", "..", "data_log", "data_kev")

# data_kev holds raw logs, so match those instead of *_cleaned.csv
csv_files = sorted(glob.glob(os.path.join(data_dir, "imu_log_*.csv")))

sensor_cols = ["AccX", "AccY", "AccZ", "GyroX", "GyroY", "GyroZ"]
smooth_cols = [col + "_smooth" for col in sensor_cols]

df_list = []

for file in csv_files:
    # Clean the same way data_eish/*_cleaned.csv was: drop repeated
    # timestamps, then apply a 5-sample moving average
    df = pd.read_csv(file).drop_duplicates(subset="Timestamp").reset_index(drop=True)
    df[smooth_cols] = df[sensor_cols].rolling(5, min_periods=1).mean()

    # Track data provenance
    df["source_file"] = os.path.basename(file)
    df_list.append(df)

# Combine all DataFrames vertically
master_df = pd.concat(df_list, ignore_index=True)

# Save the master dataset in the current directory (master_datasets)
output_path = "kev_master.csv"
master_df.to_csv(output_path, index=False)

print(f"Successfully combined {len(csv_files)} files into '{output_path}'.")
print(f"Master Dataset Shape: {master_df.shape}")

data cleaning, inspect structure

In [ ]:
import pandas as pd
import numpy as np

# Load the dataset
file_path = "kev_master.csv"  # Update path if running outside the kevin folder
df = pd.read_csv(file_path)

# Display basic information
print("--- Initial Overview ---")
print(f"Dataset Shape: {df.shape}")
print("\n--- Data Types & Missing Values ---")
print(df.info())
print("\n--- Missing Values Count ---")
print(df.isnull().sum())

df.head()

In [ ]:
# 1. Remove duplicate rows (ignoring pure identical duplicates)
initial_rows = len(df)
df = df.drop_duplicates()
print(f"Removed {initial_rows - len(df)} duplicate rows.")

# 2. Handle missing values
# Option A: Forward-fill missing sensor values (common for time-series IMU data)
# Option B: Drop rows with missing values if critical
numeric_cols = df.select_dtypes(include=[np.number]).columns
df[numeric_cols] = df[numeric_cols].ffill().bfill()

# Check remaining missing values
print(f"Remaining nulls: {df.isnull().sum().sum()}")

In [ ]:
# Assuming there is a timestamp column (e.g., 'timestamp', 'time', or 'date')
# Adjust column name if different in your dataset
timestamp_col = [col for col in df.columns if 'time' in col.lower() or 'date' in col.lower()]

if timestamp_col:
    col_name = timestamp_col[0]
    df[col_name] = pd.to_datetime(df[col_name])
    df = df.sort_values(by=col_name).reset_index(drop=True)
    print(f"Parsed and sorted by timestamp column: '{col_name}'")
else:
    print("No timestamp column detected; skipping time sorting.")

In [ ]:
# Identify numeric columns excluding metadata like source_file
features = df.select_dtypes(include=[np.number]).columns

for col in features:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    # Clip extreme values to the upper and lower threshold limits
    df[col] = df[col].clip(lower=lower_bound, upper=upper_bound)

print("Outlier clipping complete using IQR method.")

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

# Select feature columns to scale (exclude categorical target labels if present)
feature_cols = df.select_dtypes(include=[np.number]).columns

df_scaled = df.copy()
df_scaled[feature_cols] = scaler.fit_transform(df[feature_cols])

print("Features successfully scaled.")
df_scaled.head()

In [ ]:
output_cleaned_path = "kev_master_preprocessed.csv"
df_scaled.to_csv(output_cleaned_path, index=False)

print(f"Preprocessed dataset successfully saved to: {output_cleaned_path}")
print(f"Final Processed Shape: {df_scaled.shape}")